# FASE 4 - JOIN

In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df_customers = pd.read_csv('../data/sales_customers.csv')
df_employees = pd.read_csv('../data/sales_employees.csv')
df_orders = pd.read_csv('../data/sales_orders.csv')
df_orderarchive = pd.read_csv('../data/sales_ordersarchive.csv')
df_products = pd.read_csv('../data/sales_products.csv')

pl_customers = pl.read_csv('../data/sales_customers.csv')
pl_employees = pl.read_csv('../data/sales_employees.csv')
pl_orders = pl.read_csv('../data/sales_orders.csv')
pl_orderarchive = pl.read_csv('../data/sales_ordersarchive.csv')
pl_products = pl.read_csv('../data/sales_products.csv')

```SQL
SELECT
    o.orderid,
    c.firstname
FROM sales.orders AS o
INNER JOIN sales.customers AS c ON c.customerid = o.customerid;
```

In [6]:
df_o = df_orders.copy()
df_c = df_customers.copy()

res_df = df_o.merge(
    df_c,
    on='customerid',
    how='inner'
)[['orderid','firstname']]

In [9]:
pl_c = pl_customers
pl_o = pl_orders

res_pl = pl_o.join(
    pl_c,
    on='customerid',
    how='inner'
).select([
    pl.col('orderid'),
    pl.col('firstname')
])

# 📝 Tu Primer Ejercicio de la Fase 4
Vamos a unir Productos con sus Categorías (aunque en tu CSV la categoría ya viene como texto, vamos a simular una unión lógica para practicar).

Tu misión: Une la tabla de Pedidos (sales_orders) con la de Productos (sales_products) para saber qué producto se vendió en cada orden.

    - Llave de unión: productid.
    - Columnas a mostrar: orderid, product (el nombre del producto) y price.

¿Cómo escribirías la versión de PostgreSQL para este cruce? (Recuerda usar alias como o para orders y p para products para que sea más limpio).

```SQL
SELECT
    o.orderid,
    p.product,
    p.price
FROM sales.orders AS o
INNER JOIN sales.products as p ON o.productid = p.productid;
```

In [11]:
df_o = df_orders.copy()
df_p = df_products.copy()

df_merge = df_o.merge(
    df_p,
    on='productid',
    how='inner'
)[['orderid','product','price']]

In [14]:
pl_o = pl_orders
pl_p = pl_products

res_pl = pl_o.join(
    pl_p,
    on='productid',
    how='inner'
).select(
    pl.col('orderid'),
    pl.col('product'),
    pl.col('price')
)

# 🔀 El siguiente nivel de la Fase 4: LEFT JOIN
Ahora que dominas el INNER JOIN (donde solo vemos lo que coincide), vamos a un caso muy común en la vida real: Queremos ver a TODOS los clientes, hayan hecho pedidos o no.

Para esto usamos el LEFT JOIN.

Tu nuevo ejercicio: Queremos un reporte de clientes que incluya:

    - Tabla Izquierda: sales.customers (Queremos ver a todos los clientes).
    - Tabla Derecha: sales.orders.
    - Columnas: firstname (del cliente) y orderid (del pedido).

El resultado esperado: Si un cliente como "Anna Adams" no ha hecho pedidos, su firstname aparecerá, pero su orderid será NULL (o None).

¿Cómo escribirías la versión de PostgreSQL para este cruce? > Pista: Cambia INNER JOIN por LEFT JOIN. Ten cuidado con quién pones en el FROM (la tabla de la izquierda).

```SQL
SELECT
    c.firstname,
    o.orderid
FROM sales.customers AS c
LEFT JOIN sales.orders AS o ON c.customerid = o.customerid;
```

In [16]:
df_c = df_customers.copy()
df_o = df_orders.copy()

df_merge = df_c.merge(
    df_o,
    on='customerid',
    how='left'
)[['firstname','orderid']]

In [18]:
pl_c = pl_customers
pl_o = pl_orders

res_pl = pl_c.join(
    pl_o,
    on='customerid',
    how='left'
).select(
    pl.col('firstname'),
    pl.col('orderid')
)

### SQL TASK 1
#### Get all customers along with their orders, but only for customers who have placed an order

```SQL
SELECT
    *
FROM sales.customers AS c
INNER JOIN sales.orders AS o ON c.customerid = o.customerid;
````

In [21]:
df_o = df_orders.copy()
df_c = df_customers.copy()

df_merge = df_c.merge(
    df_o,
    on = 'customerid',
    how = 'inner'
)

In [23]:
pl_c = pl_customers
pl_o = pl_orders

pl_join = pl_c.join(
    pl_o,
    on = 'customerid',
    how = 'inner'
)

### SQL TASK 2

#### Get all customers along with their orders, including those without orders

```SQL
SELECT
    c.customerid,
    c.firstname,
    o.orderid,
    o.sales
FROM sales.customers AS c
LEFT JOIN sales.orders AS o ON c.customerid = o.customerid;
```

In [26]:
df_c = df_customers.copy()
df_o = df_orders.copy()

df_merge = df_c.merge(
    df_o,
    on = 'customerid',
    how = 'left'
)[['customerid','firstname','orderid','sales']]

### SQL TASK

#### Using SalesDB, Retrieve a list of all orders, along with the related customer, product, and employee details

**For each order, desplay**
- Order Id
- Customer's name
- Product name
- Sales amount
- Product price
- Salesperson's name

```SQL
SELECT
    o.orderid,
    o.sales,
    c.firstname AS CustomerFirstNmae,
    c.lastname AS CustomerLastName,
    p.product AS ProductName,
    p.price,
    e.firstname AS EmployeeFirstName,
    e.lastname AS EmployeeLastName
FROM sales.orders AS o
LEFT JOIN sales.customers AS c ON c.customerid = o.customerid
LEFT JOIN sales.products AS p ON p.productid = o.productid
LEFT JOIN sales.employees AS e on e.employeeid = o.salespersonid;
```

In [36]:
df_o = df_orders.copy()
df_c = df_customers.copy()
df_p = df_products.copy()
df_e = df_employees.copy()

df_final = (
    df_o.merge(df_c, on='customerid',how='left')
        .merge(df_p, on='productid', how='left')
        .merge(df_e, left_on='salespersonid', right_on='employeeid', how='left')
)

res_pandas = df_final[[
    'orderid',
    'sales',
    'firstname_x',
    'lastname_x',
    'product',
    'price',
    'firstname_y',
    'lastname_y'
]].rename(columns={
    'firstname_x': 'CustomerFirstName',
    'lastname_x': 'CustomerLastname',
    'firstname_y': 'EmployeeFirstName',
    'lastname_y': 'EmployeeLastName',
    'product': 'ProductName'
})

In [42]:
pl_o = pl_orders
pl_c = pl_customers
pl_p = pl_products
pl_e = pl_employees

pl_join = (
    pl_o.join(pl_c, on='customerid', how='left')
        .join(pl_p, on='productid', how='left')
        .join(pl_e, left_on='salespersonid', right_on='employeeid', how='left')
).select([
    pl.col('orderid'),
    pl.col('sales'),
    pl.col('firstname').alias('CustomerFirstName'),
    pl.col('lastname').alias('CustomerLastName'),
    pl.col('product').alias('ProductName'),
    pl.col('price'),
    pl.col('firstname_right').alias('EmployeeFirstName'),
    pl.col('lastname_right').alias('EmployeeLastname')
])